# Lab Day 2 — DeepWeeds | Doan Quang Thang — 2A202602395

Notebook chạy đầy đủ bài lab, không chứa số liệu giả. Mọi lựa chọn dựa trên val; chỉ mở test sau khi chốt cấu hình.

**Code tải trực tiếp bằng git clone**, cố định commit chứa code đã sửa và kết quả phiên bản 5 được bảo toàn. Mặc định `RUN_EXPERIMENTS = False` để xem kết quả đã hoàn tất. Đổi thành `True` khi muốn chạy mới từ ảnh/CSV công khai, không cần dataset riêng tư. `RESUME_FROM_INPUT = True` chỉ khi chủ động gắn output cũ để phục hồi; thiếu input sẽ dừng.

**Luồng:** kiểm tra dữ liệu → 5 backbone → 3 trục huấn luyện và EMA/kết hợp → 6 phương pháp suy luận ngoài mốc (gộp BN không áp dụng cho ConvNeXt) → chung kết và mốc với seed 0/1/2 → Excel, biểu đồ, báo cáo và đánh giá bằng eval.py gốc.

Bật **GPU** và **Internet**. Cấu hình nền 10 epoch, batch 32 cho mọi backbone. Chạy toàn bộ có thể cần nhiều giờ; thời gian thực được lưu theo từng epoch. Không thay đổi cấu hình sau khi đã xem test.

Nguồn: https://github.com/conanWinner/K4-DAY02-DoanQuangThang-2A202602395


## 0. Clone Git và chọn chế độ chạy
Code Python nằm trực tiếp trong repo, không có PAYLOAD. Chỉ cài thư viện khi RUN_EXPERIMENTS = True.

In [ ]:
import hashlib, json, sys, subprocess
from pathlib import Path

# False: open verified results from version 5. True: install dependencies and run experiments.
RUN_EXPERIMENTS = False
RESUME_FROM_INPUT = False  # Optional: attach prior output; never silently fall back to retraining.
RUN_CHECKS = True
REPO_URL = 'https://github.com/conanWinner/K4-DAY02-DoanQuangThang-2A202602395.git'
CODE_REVISION = 'd3b7392043b5f3bb3afcf0a88cc96a70766ff891'
REPO_DIR = Path('/kaggle/working/deepweeds_repo_' + CODE_REVISION[:12])
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', CODE_REVISION], check=True)
actual_revision = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if actual_revision != CODE_REVISION:
    raise RuntimeError('Existing repository has another revision; preserve it and use a new directory')
if subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip():
    raise RuntimeError('Existing repository has local changes; preserve them before running')
SUBMISSION_DIR = REPO_DIR / 'submissions/2A202602395_DoanQuangThang'
SOURCE_DIR = SUBMISSION_DIR / 'code'
sys.path[:0] = [str(SOURCE_DIR), str(REPO_DIR)]
if RUN_EXPERIMENTS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SOURCE_DIR / 'requirements.txt')], check=True)
print('Git revision:', actual_revision)
print('Code directory:', SOURCE_DIR)
print('Run experiments:', RUN_EXPERIMENTS)


## 1. Kiểm tra môi trường và tính đúng của code
Các fixture tổng hợp chỉ dùng cho kiểm tra, không đưa vào báo cáo DeepWeeds.

In [ ]:
if RUN_CHECKS:
    # Standard-library regression tests only: no training or CUDA required.
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(SOURCE_DIR / 'tests'),
                    '-p', 'test_final_alias.py', '-v'], check=True, cwd=REPO_DIR)


In [ ]:
if RUN_EXPERIMENTS:
    import importlib.metadata as metadata, platform
    import torch
    from deepweeds_lab.train import write_json
    OUTPUT_DIR = Path('/kaggle/working/lab_output')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    from deepweeds_lab.recovery import restore_kaggle_input
    # Fresh runs require only public image/CSV sources. Recovery is an explicit optional mode.
    if RESUME_FROM_INPUT:
        restore_kaggle_input(OUTPUT_DIR, required=True)
    else:
        print('Fresh run: official fold 0; no private recovery input required')
    versions = {name: metadata.version(name) for name in ('torch', 'torchvision', 'timm', 'numpy', 'pandas', 'scipy', 'matplotlib', 'openpyxl', 'fvcore')}
    versions.update(python=platform.python_version(), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
    write_json(OUTPUT_DIR / 'environment.json', versions)
    print(versions, flush=True)
    if not torch.cuda.is_available():
        raise RuntimeError('Full lab requires GPU; enable an accelerator before running')
    # CPU arithmetic checks and a tiny synthetic train only; not DeepWeeds results.
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(SOURCE_DIR / 'tests'), '-v'], check=True, cwd=SOURCE_DIR)
else:
    print('Verified version 5 results: no new training requested')


## 2–5. Chạy các vòng thí nghiệm

- B01–B05: ResNet-50, ResNeXt-50, ConvNeXt-Tiny, DeiT-Small, MobileNetV3-Large; cùng nền và seed.
- T00–T09: khởi tạo, augmentation, loss; thêm EMA và một kết hợp. Mỗi thí nghiệm riêng chỉ đổi một trục.
- I00–I07: một ảnh, lật/gộp xác suất, lật/gộp logit, 5 crop, độ phân giải 256, temperature scaling, AMP và gộp BN.
- F01/T00: cấu hình cuối và nền với ≥3 seed; R01 nếu có phương pháp p95 ≤100 ms. Nhiệt độ khớp trên val của từng seed trước test. Mỗi cấu hình/seed dùng một lần forward trên test (TTA gồm các view đã khai báo); tái sử dụng logits cho so sánh hiệu chuẩn.
- Báo cáo sinh từ log, prediction CSV; eval.py gốc tính lại toàn bộ test.


In [ ]:
if RUN_EXPERIMENTS:
    from deepweeds_lab.experiments import execute
    # All required stages; test is locked until backbone/training/inference selection completes.
    # Kaggle output retains per-epoch last.pt/best.pt/history.csv. Reuse the same configuration
    # and restored lab_output to resume after interruption; never reselect from test results.
    try:
        selection = execute(str(OUTPUT_DIR), epochs=10, batch_size=32, seeds=(0, 1, 2))
    except Exception as exc:
        import traceback
        failure = {'status': 'error', 'type': type(exc).__name__, 'message': str(exc), 'traceback': traceback.format_exc()}
        write_json(OUTPUT_DIR / 'execution_error.json', failure)
        raise
else:
    print('Training skipped; displaying completed version 5 report')


## 6. Xem kết quả thật
Chế độ xem dùng kết quả phiên bản 5 đã lưu trên Git; chế độ chạy dùng lab_output mới. Đường dẫn file được in bên dưới.

In [ ]:
from IPython.display import display, Markdown
RESULT_DIR = OUTPUT_DIR if RUN_EXPERIMENTS else SUBMISSION_DIR
STATUS_FILE = RESULT_DIR / 'execution_status.json' if RUN_EXPERIMENTS else RESULT_DIR / 'logs/execution_status.json'
print('Result origin:', 'current execution' if RUN_EXPERIMENTS else 'completed Kaggle version 5, archived in Git')
print(STATUS_FILE.read_text())
print('Excel:', RESULT_DIR / 'results.xlsx')
print('Prediction files:', len(list((RESULT_DIR / 'predictions').glob('*.csv'))))


In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)


## 6.1. So sánh backbone
5 backbone, cùng công thức nền; chọn theo validation.

In [ ]:
display(pd.read_excel(RESULT_DIR / 'results.xlsx', sheet_name='Backbones'))


## 6.2. Công thức huấn luyện
Các trục khởi tạo, augmentation, loss và các thử nghiệm bổ sung.

In [ ]:
display(pd.read_excel(RESULT_DIR / 'results.xlsx', sheet_name='Training'))


## 6.3. Phương pháp suy luận
Chất lượng trên validation, hiệu chuẩn và chi phí suy luận.

In [ ]:
display(pd.read_excel(RESULT_DIR / 'results.xlsx', sheet_name='Inference'))


## 6.4. Kết quả cuối qua ba seed
Kết quả test sau khi chốt cấu hình; có dòng tổng hợp mean và std.

In [ ]:
display(pd.read_excel(RESULT_DIR / 'results.xlsx', sheet_name='Final'))


## 6.5. Chỉ số từng lớp
Precision, recall và F1 của cấu hình cuối và mốc.

In [ ]:
display(pd.read_excel(RESULT_DIR / 'results.xlsx', sheet_name='PerClass'))


## 6.6. Độ trễ
p50/p95/p99 và thông lượng theo batch, GPU và dtype.

In [ ]:
display(pd.read_excel(RESULT_DIR / 'results.xlsx', sheet_name='Latency'))


## 6.7. Tổng hợp cấu hình
Top 10 và mốc T00; nguồn/loại độ trễ được ghi rõ. Chưa đo riêng không được điền số suy đoán.

In [ ]:
display(pd.read_excel(RESULT_DIR / 'results.xlsx', sheet_name='Summary'))


## 6.8. Biểu đồ và ảnh phân tích
Ảnh lấy từ kết quả thật của phiên chạy tương ứng.

In [ ]:
from IPython.display import Image
for filename in ('class_distribution.png', 'accuracy_latency.png', 'confusion_matrix.png', 'misclassified_images.png'):
    print(filename)
    display(Image(filename=str(RESULT_DIR / filename)))


## 7. Báo cáo đầy đủ
Nội dung gốc sinh từ log chạy thật, gồm hạn chế và phân tích kết quả.

In [ ]:
display(Markdown((RESULT_DIR / 'report.md').read_text()))
